# Упражнение 05

## Пункт первый

Прочитай файл fines.csv, который ты сохранил(а) в предыдущем упражнении.
Во всех подпунктах далее тебе нужно вычислять fines/refund*year для каждой строки и создать новый столбец с рассчитанными данными.
Измеряй время с помощью магической команды %timeit в ячейке.

In [1]:
import pandas as pd
import gc

fines = pd.read_csv("../ex04/data/fines.csv")

## Напиши функцию, которая проходит по DataFrame циклом for i in range(0, len(df)), использует iloc и append() в список. Результат функции присвой новому столбцу в DataFrame

In [2]:
%%timeit

def data_frames_iter(frame):
    result = []
    for i in range(0, len(frame)):
        row = frame.iloc[i]
        if row["Refund"] == 0:
            res = 0
        else:
            res = row["Fines"]/row["Refund"]*row["Year"]
        result.append(res)

    return result

fines["FinesPerRefund"] = data_frames_iter(fines)
fines


20.5 ms ± 3.62 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Сделай то же с использованием iterrows().


In [3]:
%%timeit

def data_frames_iterrow(frame):
    result = []
    for index, row in frame.iterrows():
        if row["Refund"] == 0:
            value = 0 
        else:
            value = row["Fines"]/row["Refund"]*row["Year"]
        result.append(value)

    return result

fines["FinesPerRefund"] = data_frames_iterrow(fines)
fines

13 ms ± 497 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Сделай то же с использованием apply() и лямбда-функции.


In [4]:
%%timeit
fines["FinesPerRefund"] = fines.apply(lambda row: row["Fines"]/row["Refund"]*row["Year"] if row["Refund"] != 0 else 0, axis=1)
fines

4.66 ms ± 248 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit
fines["FinesPerRefund"] = (fines["Fines"].div(fines["Refund"].replace(0, pd.NA)).mul(fines["Year"]).fillna(0))
fines

201 μs ± 1.88 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Сделай как в предыдущем подпункте, но используй метод .values

In [6]:
%%timeit
fines_values = fines["Fines"].values
refund_values = fines["Refund"].values
year_values = fines["Year"].values

result = fines_values.astype(float) * 0
mask = refund_values != 0

result[mask] = (fines_values[mask] / refund_values[mask] * year_values[mask])

fines["FinesPerRefund"] = result
fines

36.4 μs ± 1.08 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Получи строку для конкретного CarNumber, например, "O136HO197RUS."

In [7]:
f = fines[fines["CarNumber"] == "O136HO197RUS"]
f

,CarNumber,Refund,Fines,Make,Model,Year,FinesPerRefund
715,O136HO197RUS,2,7800.0,Toyota,Corolla,1999,7796100.0
902,O136HO197RUS,2,7800.0,Toyota,Corolla,1991,7764900.0


## Установи индекс в своём DataFrame столбцом CarNumber.

In [8]:
fines = fines.set_index("CarNumber")
fines.index

Index(['Y163O8161RUS', 'E432XX77RUS', '7184TT36RUS', 'X582HE161RUS',
       '92918M178RUS', 'H234YH197RUS', 'E40577152RUS', '707987163RUS',
       'K330T8197RUS', 'X786CO96RUS',
       ...
       '8182XX154RUS', 'X796TH96RUS', 'T011MY163RUS', 'T341CC96RUS',
       'T119CT96RUS', 'A001AA77RUS', 'B002BB77RUS', 'C003CC77RUS',
       'E004EE77RUS', 'K005KK77RUS'],
      dtype='str', name='CarNumber', length=930)

## Снова получи строку для того же CarNumber.

In [9]:
%%timeit
s = fines[fines.index == 'O136HO197RUS']
s

99.2 μs ± 2.31 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Выполни df.info(memory_usage='deep') и обрати внимание на Dtype и объём памяти.

In [ ]:
fines.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to K005KK77RUS
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Refund          930 non-null    int64  
 1   Fines           930 non-null    float64
 2   Make            930 non-null    str    
 3   Model           919 non-null    str    
 4   Year            930 non-null    int64  
 5   FinesPerRefund  930 non-null    float64
dtypes: float64(2), int64(2), str(2)
memory usage: 182.0 KB
<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to K005KK77RUS
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Refund          930 non-null    int64  
 1   Fines           930 non-null    float64
 2   Make            930 non-null    str    
 3   Model           919 non-null    str    
 4   Year            930 non-null    int64  
 5   FinesPerRefund  930 non-null    float64
dtypes

## Создай copy() исходного DataFrame в другой DataFrame - optimized_df.


In [11]:

optimized_df = fines.copy()

## Понизь разрядность типа с float64 до float32 для всех столбцов.
## Понизь разрядность типа с int64 до минимально возможного числового Dtype.


In [12]:
optimized_df["Fines"] = optimized_df["Fines"].astype("float32")
optimized_df["FinesPerRefund"] = optimized_df["FinesPerRefund"].astype("float32")
optimized_df["Refund"] = pd.to_numeric(optimized_df["Refund"], downcast="integer")
optimized_df["Year"] = pd.to_numeric(optimized_df["Year"], downcast="integer")
optimized_df.dtypes

Refund               int8
Fines             float32
Make                  str
Model                 str
Year                int16
FinesPerRefund    float32
dtype: object

## Запусти info(memory_usage='deep') для нового DataFrame и обрати внимание на Dtype и объём памяти.


In [13]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to K005KK77RUS
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Refund          930 non-null    int8   
 1   Fines           930 non-null    float32
 2   Make            930 non-null    str    
 3   Model           919 non-null    str    
 4   Year            930 non-null    int16  
 5   FinesPerRefund  930 non-null    float32
dtypes: float32(2), int16(1), int8(1), str(2)
memory usage: 162.9 KB


## Преобразуй столбцы типа object в category.

На этот раз проверь объём памяти. Вероятно, он уменьшится в 2-3 раза по сравнению с исходным DataFrame.

In [14]:
object_columns = optimized_df.select_dtypes(include="object").columns
optimized_df[object_columns] = optimized_df[object_columns].astype("category")

optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to K005KK77RUS
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Refund          930 non-null    int8    
 1   Fines           930 non-null    float32 
 2   Make            930 non-null    category
 3   Model           919 non-null    category
 4   Year            930 non-null    int16   
 5   FinesPerRefund  930 non-null    float32 
dtypes: category(2), float32(2), int16(1), int8(1)
memory usage: 68.2 KB


/var/folders/3h/9_g0v8b11knbvn8ty7rfvrzw0000gn/T/ipykernel_62997/2607599098.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = optimized_df.select_dtypes(include="object").columns


## Используя библиотеку gc и команду %reset_selective, удали из памяти только свой исходный DataFrame

In [15]:
%reset_selective -f ^fines$

gc.collect()
optimized_df


,Refund,Fines,Make,Model,Year,FinesPerRefund
CarNumber,,,,,,
Y163O8161RUS,2,3200.0,Ford,Focus,1989,3182400.0
E432XX77RUS,1,6500.0,Toyota,Camry,1995,12967500.0
7184TT36RUS,1,2100.0,Ford,Focus,1984,4166400.0
X582HE161RUS,2,2000.0,Ford,Focus,2015,2015000.0
92918M178RUS,1,5700.0,Ford,Focus,2014,11479800.0
...,...,...,...,...,...,...
A001AA77RUS,1,1500.0,Toyota,Camry,2015,3022500.0
B002BB77RUS,0,2500.0,BMW,X5,2018,0.0
C003CC77RUS,2,500.0,Ford,Focus,2012,503000.0


In [16]:
fines

NameError: name 'fines' is not defined